In [1]:
import sys
from pathlib import Path
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\examples')
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as ag
assert ag.__version__ == '2.1.3'
assert 'site-packages' in Path(ag.__file__).parts


# Tools y Skills

Input 3 → Tool increment resuelta en la Skill → output 4.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as ag


In [3]:
@ag.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@ag.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = ag.skill(name="arithmetic", tools=[increment, double])
increase = ag.Action("increase", "increment")
multiply = ag.Action("multiply", "double")

In [4]:
agent = ag.Agent(name="incrementer", flow=increase, skills=(arithmetic,))
result = agent.run({"value": 3})
assert result.ok and result.data == {"value": 4}
print({"input": {"value": 3}, "process": "Skill arithmetic → increment", "output": result.data})

{'input': {'value': 3}, 'process': 'Skill arithmetic → increment', 'output': {'value': 4}}


Una Skill no se ejecuta. Se asocia con `skills=(arithmetic,)`; Action resuelve la Tool por nombre. `load_skill(path)` carga una Skill portable escrita con esta misma API; requiere `SKILL.md` y `skill.py` con `build_skill()`. Es carga desde archivo, no una segunda gramática de construcción.

Los parámetros canónicos de contrato son input_schema y output_schema. Este ejemplo usa modelos Pydantic con StrictInt. Tool expone run, arun e inspect; no tiene compile. Una Skill expone instrucciones y capacidades, no otro ejecutor.

In [5]:
from tool_contracts.example import increment as validated
assert validated.run({'value':3}).data == {'value':4}
assert (await validated.arun({'value':3})).data == {'value':4}
assert not validated.run({'value':'invalid'}).ok
print(validated.inspect())

{'name': 'increment', 'description': 'Increment an explicitly typed integer.', 'metadata': {'example': 'tool_contracts'}, 'input_schema': {'properties': {'value': {'title': 'Value', 'type': 'integer'}}, 'required': ['value'], 'title': 'Quantity', 'type': 'object'}, 'output_schema': {'properties': {'value': {'title': 'Value', 'type': 'integer'}}, 'required': ['value'], 'title': 'Quantity', 'type': 'object'}}


In [6]:
from portable_skills.example import run as portable_example
report = portable_example()
print({'input':report['input'], 'process':report['process'], 'output':report['output']['data']})

{'input': {'value': 3}, 'process': {'name': 'arithmetic', 'description': 'Portable local arithmetic', 'version': '1.0.0', 'instructions': 'Use exact integer arithmetic.', 'tools': [{'name': 'increment', 'description': 'Increment an integer using a sibling resource.', 'metadata': {}, 'input_schema': None, 'output_schema': None}], 'metadata': {'synthetic': True}}, 'output': {'value': 4}}


In [7]:
import tempfile
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder)
    (path / "SKILL.md").write_text("# Arithmetic", encoding="utf-8")
    (path / "skill.py").write_text("import agentic_systems as ag\n@ag.tool\ndef inc(value:int)->dict:\n    return {'value':value+1}\ndef build_skill():\n    return ag.skill(name='loaded',tools=[inc])\n", encoding="utf-8")
    loaded = ag.load_skill(path)
    loaded_agent = ag.Agent("loaded", ag.Action("inc", "inc"), skills=(loaded,))
    assert loaded_agent.run({"value": 3}).data == {"value": 4}
print("Skill portable cargada y ejecutada mediante Agent")

Skill portable cargada y ejecutada mediante Agent


In [8]:
assert blocked_attempts() == 0